# Online Learning Engagement Prediction & Early Warning Alert System

## 📖 Project Overview
In digital and distance education platforms (such as Coursera, edX, Canvas, and university LMS portals), identifying disengaged and at-risk students before they drop out is critical for educational outcomes.

This end-to-end data science project covers:
1. **Data Source & Telemetry Simulation**: Ingestion of multi-modal online student interactions (logins, video completion, quiz performance, assignments, forum posts, inactivity).
2. **Exploratory Data Analysis (EDA)**: Statistical distributions, outlier analysis, correlation heatmaps, and behavioral patterns.
3. **Data Preprocessing & Cleaning**: Missing value imputation, categorical encoding, and domain-specific feature engineering.
4. **Feature Scaling**: Standardization with `StandardScaler`.
5. **Supervised Machine Learning**: Training and comparing Random Forest, Gradient Boosting, Logistic Regression, SVM, and KNN.
6. **Evaluation & Explainability**: Confusion matrices, F1-scores, ROC-AUC curves, and feature importance rankings.
7. **Actionable Alert System**: Risk score calculation and pedagogical intervention recommendations.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Add src to path
sys.path.append(os.path.abspath(".."))
from src.data_generator import generate_engagement_data
from src.preprocessing import prepare_data, engineer_features
from src.model_training import train_and_evaluate_models
from src.alert_system import EngagementAlertEngine

## 1. 📂 Data Ingestion & Overview

In [ ]:
raw_data_path = "../data/raw_online_learning_engagement.csv"
if os.path.exists(raw_data_path):
    df = pd.read_csv(raw_data_path)
else:
    df = generate_engagement_data(n_samples=3000, random_seed=42)
    os.makedirs("../data", exist_ok=True)
    df.to_csv(raw_data_path, index=False)

print(f"Total student records: {len(df)}")
display(df.head())
display(df.info())

## 2. 📊 Exploratory Data Analysis & Visualizations

Let's examine the target class distribution and relationships between student interaction metrics and their engagement outcomes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Target distribution
engagement_counts = df['engagement_level'].value_counts()
palette = {'Low': '#ef4444', 'Medium': '#f59e0b', 'High': '#10b981'}
sns.barplot(x=engagement_counts.index, y=engagement_counts.values, ax=axes[0], palette=palette, hue=engagement_counts.index, legend=False)
axes[0].set_title("Distribution of Student Engagement Levels", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Number of Students")

# 2. Inactivity vs Engagement
sns.boxplot(x='engagement_level', y='days_inactive_last_30_days', data=df, ax=axes[1], palette=palette, hue='engagement_level', legend=False)
axes[1].set_title("Days Inactive in Past Month vs. Engagement Level", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Days Inactive")

plt.tight_layout()
plt.show()

### Telemetry Interaction Insights
From the boxplot above, students with **Low Engagement (At-Risk)** exhibit significantly higher days of platform dormancy (typically > 12 days in the last month), whereas high engagement students rarely remain inactive for more than 4 days.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x='video_watch_time_hours',
    y='quiz_avg_score',
    hue='engagement_level',
    size='login_frequency_per_week',
    palette=palette,
    alpha=0.7
)
plt.title("Video Watch Time vs. Quiz Score by Engagement Class", fontsize=14, fontweight='bold')
plt.xlabel("Video Watch Time (Hours/Month)")
plt.ylabel("Average Quiz Score (%)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. ⚙️ Preprocessing, Feature Engineering & Scaling

We engineer three domain-specific composite metrics:
1. `active_days_ratio`: Proportion of active learning days per month.
2. `study_intensity_score`: Weekly study load (Logins × Session Length / 60).
3. `forum_participation_rate`: Peer interaction intensity per active login.

In [ ]:
data_dict = prepare_data(data_path=raw_data_path)
print("Transformed Feature Matrix Shape:", data_dict['X_train_transformed'].shape)
print("Engineered Features:", data_dict['feature_names'])

## 4. 🤖 Multi-Model Training & Evaluation Leaderboard

In [ ]:
metadata, best_model = train_and_evaluate_models(data_dict, save_dir="../models")
comparison_df = pd.DataFrame.from_dict(metadata['model_comparison'], orient='index')
display(comparison_df.sort_values(by='f1_macro', ascending=False))

## 5. 🔍 Model Explainability & Feature Importance

In [ ]:
feat_imp = pd.Series(metadata['feature_importance']).head(10)
plt.figure(figsize=(10, 5))
feat_imp.sort_values().plot(kind='barh', color='#0284c7')
plt.title(f"Top 10 Feature Importances - Best Model ({metadata['best_model_name']})", fontsize=13, fontweight='bold')
plt.xlabel("Relative Importance Score")
plt.tight_layout()
plt.show()

## 6. 🚨 Early-Warning Alert Engine Demonstration

In [ ]:
alert_engine = EngagementAlertEngine(models_dir="../models")

test_learner = {
    'course_category': 'Data Science',
    'device_type': 'Desktop',
    'login_frequency_per_week': 2,
    'video_watch_time_hours': 3.2,
    'video_completion_rate': 0.20,
    'quiz_attempts': 1,
    'quiz_avg_score': 38.0,
    'assignment_submission_rate': 0.25,
    'discussion_forum_posts': 0,
    'days_inactive_last_30_days': 18,
    'time_spent_per_session_mins': 15.0,
    'previous_course_gpa': 2.3
}

diag = alert_engine.predict_single_student(test_learner)
print("=" * 50)
print(f"🎯 Prediction: {diag['predicted_engagement_level']}")
print(f"🚨 Alert Status: {diag['alert_badge']}")
print(f"📊 Disengagement Risk Score: {diag['disengagement_risk_score'] * 100:.1f}%")
print("=" * 50)
print("Recommended Pedagogical Actions:")
for act in diag['recommended_interventions']:
    print(" ->", act)